# 05 Feature Experimentation

Goal: compare raw and engineered feature sets using one fixed split, one fixed CV strategy, and one fixed Random Forest configuration. The test split is used only after selecting the best feature set from cross-validation.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

from src.features import (
    BASIC_APPLICATION_FEATURES,
    ENGINEERED_FEATURE_GROUPS,
    create_features,
)
from src.preprocessing import build_preprocessor
from src.evaluation import compare_feature_set_cv, evaluate_classifier


In [2]:
RANDOM_STATE = 42
DOMINANCE_THRESHOLD = 0.995

RF_PARAMS = {
    "n_estimators": 200,
    "min_samples_split": 2,
    "min_samples_leaf": 10,
    "max_features": "log2",
    "max_depth": None,
    "class_weight": "balanced_subsample",
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE,
)


def make_rf_model():
    return RandomForestClassifier(**RF_PARAMS)


def find_dominant_columns(X_train, threshold=DOMINANCE_THRESHOLD):
    dominant_columns = []

    for column in X_train.columns:
        dominant_ratio = (
            X_train[column]
            .value_counts(normalize=True, dropna=False)
            .iloc[0]
        )

        if dominant_ratio > threshold:
            dominant_columns.append(column)

    return dominant_columns


def evaluate_feature_matrix(name, X_train, y_train):
    return compare_feature_set_cv(
        name=name,
        X=X_train,
        y=y_train,
        model=make_rf_model(),
        cv=cv,
        scoring="roc_auc",
    )


def build_pipeline(X_train):
    numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
    categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()

    preprocessor = build_preprocessor(
        numeric_columns=numeric_columns,
        categorical_columns=categorical_columns,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", make_rf_model()),
        ]
    )


In [3]:
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

train_df = pd.read_csv(
    RAW_DATA_DIR / "application_train.csv"
)

y = train_df["TARGET"].copy()

train_index, test_index = train_test_split(
    train_df.index,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

y_train = y.loc[train_index]
y_test = y.loc[test_index]

train_df.shape, y_train.mean(), y_test.mean()


((307511, 122),
 np.float64(0.08072908198107379),
 np.float64(0.08072776937710356))

In [4]:
X_all_raw = train_df.drop(
    columns=["TARGET", "SK_ID_CURR"]
).copy()

X_all_engineered = create_features(X_all_raw)

curated_columns = [
    column
    for column in BASIC_APPLICATION_FEATURES
    if column in train_df.columns
]

X_curated_raw = train_df[curated_columns].copy()
X_curated_engineered = create_features(X_curated_raw)

base_feature_matrices = {
    "Curated raw": X_curated_raw,
    "Curated raw + engineered": X_curated_engineered,
    "All raw": X_all_raw,
    "All raw + engineered": X_all_engineered,
}

{
    name: matrix.shape[1]
    for name, matrix in base_feature_matrices.items()
}


{'Curated raw': 17,
 'Curated raw + engineered': 32,
 'All raw': 120,
 'All raw + engineered': 149}

In [5]:
feature_sets = {}
dominance_summary = []

for name, matrix in base_feature_matrices.items():
    X_train_base = matrix.loc[train_index].copy()
    X_test_base = matrix.loc[test_index].copy()

    feature_sets[name] = {
        "X_train": X_train_base,
        "X_test": X_test_base,
        "dropped_columns": [],
    }

    dominant_columns = find_dominant_columns(X_train_base)

    dominance_summary.append({
        "feature_set": name,
        "dropped_column_count": len(dominant_columns),
        "dropped_columns": dominant_columns,
    })

    feature_sets[f"{name}, drop dominance >99.5%"] = {
        "X_train": X_train_base.drop(columns=dominant_columns, errors="ignore"),
        "X_test": X_test_base.drop(columns=dominant_columns, errors="ignore"),
        "dropped_columns": dominant_columns,
    }

pd.DataFrame(dominance_summary)[
    ["feature_set", "dropped_column_count"]
]


,feature_set,dropped_column_count
0,Curated raw,0
1,Curated raw + engineered,0
2,All raw,16
3,All raw + engineered,16


In [6]:
experiment_results = []

for name, matrices in feature_sets.items():
    print(f"Evaluating: {name}")

    result = evaluate_feature_matrix(
        name=name,
        X_train=matrices["X_train"],
        y_train=y_train,
    )

    experiment_results.append(result)

results_df = pd.DataFrame(experiment_results)

all_raw_auc = results_df.loc[
    results_df["feature_set"] == "All raw",
    "mean_cv_auc",
].iloc[0]

results_df["delta_vs_all_raw"] = (
    results_df["mean_cv_auc"] - all_raw_auc
)

results_df = results_df.sort_values(
    "mean_cv_auc",
    ascending=False,
).reset_index(drop=True)

results_df


Evaluating: Curated raw


Evaluating: Curated raw, drop dominance >99.5%


Evaluating: Curated raw + engineered


Evaluating: Curated raw + engineered, drop dominance >99.5%


Evaluating: All raw


Evaluating: All raw, drop dominance >99.5%


Evaluating: All raw + engineered


Evaluating: All raw + engineered, drop dominance >99.5%


,feature_set,raw_feature_count,transformed_feature_count,mean_cv_auc,std_cv_auc,minimum_auc,maximum_auc,delta_vs_all_raw
0,"Curated raw + engineered, drop dominance >99.5%",32,92,0.745732,0.001408,0.744304,0.747647,0.036610
1,Curated raw + engineered,32,92,0.745732,0.001408,0.744304,0.747647,0.036610
2,"Curated raw, drop dominance >99.5%",17,63,0.740525,0.001585,0.739368,0.742767,0.031403
3,Curated raw,17,63,0.740525,0.001585,0.739368,0.742767,0.031403
4,"All raw + engineered, drop dominance >99.5%",133,339,0.738945,0.001407,0.737362,0.740780,0.029823
5,All raw + engineered,149,355,0.738677,0.001085,0.737620,0.740168,0.029555
6,"All raw, drop dominance >99.5%",104,289,0.712014,0.002534,0.708575,0.714606,0.002892
7,All raw,120,305,0.709122,0.002773,0.705262,0.711650,0.000000


In [7]:
best_feature_set = results_df.loc[0, "feature_set"]
best_train = feature_sets[best_feature_set]["X_train"]

ablation_results = []

baseline_result = evaluate_feature_matrix(
    name=best_feature_set,
    X_train=best_train,
    y_train=y_train,
)

baseline_auc = baseline_result["mean_cv_auc"]
ablation_results.append({
    "removed_group": "none",
    "mean_cv_auc": baseline_auc,
    "auc_change_vs_baseline": 0.0,
    "removed_feature_count": 0,
})

for group_name, columns in ENGINEERED_FEATURE_GROUPS.items():
    existing_columns = [
        column
        for column in columns
        if column in best_train.columns
    ]

    if not existing_columns:
        continue

    X_candidate = best_train.drop(columns=existing_columns)

    result = evaluate_feature_matrix(
        name=f"{best_feature_set} without {group_name}",
        X_train=X_candidate,
        y_train=y_train,
    )

    ablation_results.append({
        "removed_group": group_name,
        "mean_cv_auc": result["mean_cv_auc"],
        "auc_change_vs_baseline": result["mean_cv_auc"] - baseline_auc,
        "removed_feature_count": len(existing_columns),
    })

ablation_df = pd.DataFrame(ablation_results).sort_values(
    "auc_change_vs_baseline"
).reset_index(drop=True)

ablation_df


,removed_group,mean_cv_auc,auc_change_vs_baseline,removed_feature_count
0,financial,0.739733,-0.005999,4
1,external,0.743824,-0.001908,5
2,time,0.744915,-0.000817,3
3,household,0.745456,-0.000276,4
4,none,0.745732,0.000000,0


In [8]:
best_X_train = feature_sets[best_feature_set]["X_train"]
best_X_test = feature_sets[best_feature_set]["X_test"]

final_pipeline = build_pipeline(best_X_train)
final_pipeline.fit(best_X_train, y_train)

final_metrics = evaluate_classifier(
    model=final_pipeline,
    X=best_X_test,
    y_true=y_test,
    threshold=0.5,
)

final_report = {
    "selected_feature_set": best_feature_set,
    "train_raw_features": best_X_train.shape[1],
    "test_roc_auc": final_metrics["roc_auc"],
    "test_precision_class_1": final_metrics["precision_class_1"],
    "test_recall_class_1": final_metrics["recall_class_1"],
    "test_f1_class_1": final_metrics["f1_class_1"],
    "test_accuracy": final_metrics["accuracy"],
}

final_report


{'selected_feature_set': 'Curated raw + engineered, drop dominance >99.5%',
 'train_raw_features': 32,
 'test_roc_auc': 0.7506813569264095,
 'test_precision_class_1': 0.2554206418039896,
 'test_recall_class_1': 0.3558912386706949,
 'test_f1_class_1': 0.29739964655390055,
 'test_accuracy': 0.8642505243646651}

In [9]:
print(final_metrics["confusion_matrix"])
print(final_metrics["classification_report"])


[[51387  5151]
 [ 3198  1767]]
              precision    recall  f1-score   support

           0       0.94      0.91      0.92     56538
           1       0.26      0.36      0.30      4965

    accuracy                           0.86     61503
   macro avg       0.60      0.63      0.61     61503
weighted avg       0.89      0.86      0.87     61503



## Interpretation Template

After running the notebook, use the tables above as follows:

- The feature-set table decides which candidate feature matrix wins under cross-validation.
- The ablation table shows which engineered groups are useful. A negative `auc_change_vs_baseline` means removing that group hurt performance, so the group helped.
- The final report evaluates the selected feature set once on the holdout test split.
